# DATA266 Lab 1 — Task 1: GPT from scratch

Run all cells in order. Select **Runtime → Change runtime type → T4 GPU** first. This notebook mounts Google Drive, reuses the shared `DATA266_Lab1_checkpoints/task1` folder, resumes from the latest checkpoint, produces the required metrics/plots/samples, validates the artifacts, and downloads a ZIP at the end.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU is not enabled. Select Runtime → Change runtime type → T4 GPU, reconnect, and rerun this cell.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile

drive.mount('/content/drive', force_remount=False)
drive_root = Path('/content/drive/MyDrive')
checkpoint_root = drive_root / 'DATA266_Lab1_checkpoints'

# The shared-folder shortcut should be directly inside My Drive. If a ZIP was shared instead, extract it automatically.
if not checkpoint_root.exists():
    zip_files = list(drive_root.rglob('DATA266_Lab1_checkpoints*.zip'))
    if zip_files:
        with zipfile.ZipFile(zip_files[0]) as archive:
            archive.extractall(drive_root)
# Use a fresh child folder so the old completed task1 checkpoint is never reused.
checkpoint_dir = checkpoint_root / 'task1_final'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
print('Checkpoint folder:', checkpoint_dir)
print('Existing checkpoints:', sorted(p.name for p in checkpoint_dir.glob('*.pt')))

!rm -rf /content/DATA266_Lab1_Denisha
!git clone https://github.com/denishatank12/GenAI.git /content/DATA266_Lab1_Denisha
%cd /content/DATA266_Lab1_Denisha
!pip install -q -r requirements.txt
!mkdir -p reproducibility/raw_logs

In [ ]:
%cd /content/DATA266_Lab1_Denisha
!python3 reproducibility/prepare_data.py --task task1 --out .

In [ ]:
%cd /content/DATA266_Lab1_Denisha/task1_llm/member_denisha
!python3 -u src/train.py --config config.yaml --checkpoint-dir /content/drive/MyDrive/DATA266_Lab1_checkpoints/task1_final --resume-latest 2>&1 | tee ../../reproducibility/raw_logs/task1_denisha.log

In [ ]:
%cd /content/DATA266_Lab1_Denisha/task1_llm/member_denisha
!python3 src/plot_history.py
from IPython.display import display, Image
display(Image('outputs/loss_curves.png'))
print(open('outputs/metrics.csv').read())
!python3 src/write_report.py
!mkdir -p checkpoints ../../reproducibility/manifests
!cp -f /content/drive/MyDrive/DATA266_Lab1_checkpoints/task1_final/gpt_from_scratch.pt checkpoints/gpt_from_scratch.pt
!python3 ../../reproducibility/collect_manifest.py --task task1 --command 'python3 -u src/train.py --config config.yaml --resume-latest' --dataset 'TinyStories character-level dataset' --checkpoint checkpoints/gpt_from_scratch.pt --out ../../reproducibility/manifests/task1_denisha.json
!python3 ../../reproducibility/validate_results.py --task task1 --root ../..

In [ ]:
%cd /content/DATA266_Lab1_Denisha/task1_llm/member_denisha
!zip -r /content/task1_denisha_artifacts.zip outputs checkpoints results.md failure_analysis.md ../../reproducibility/raw_logs/task1_denisha.log ../../reproducibility/manifests/task1_denisha.json
from google.colab import files
files.download('/content/task1_denisha_artifacts.zip')